In [4]:
!pip install --force-reinstall pandas

  Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.1-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl (9.8 MB)
Using cached numpy-2.5.1-cp313-cp313-win_amd64.whl (12.4 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

  Attempting uninstall: tzdata

    Found existing installation: tzdata 2026.2

   ---------------------------------------- 0/5 [tzdata]
   ---------------------------------------- 0/5 [tzdata]
    Uninstalling tzdata-2026.2:
   ---------------------------------------- 0/5 [tzdata]
   ---------------------------------------- 0/5 [tzdata]

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'D:\\New folder\\Lib\\site-packages\\numpy\\linalg\\_umath_linalg.cp313-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# ==========================================
# STEP 1: LOAD DATASET
# ==========================================
print("--- STEP 1: LOADING DATASET ---")
df = pd.read_csv("SCMS Delivery History Dataset (1).csv")
print(f"Dataset Shape: {df.shape}")

# ==========================================
# STEP 2: DATA CLEANING & ETL
# ==========================================
print("\n--- STEP 2: EXECUTING DATA CLEANING ---")

# 1. Convert Date columns to datetime format
date_columns = ['Scheduled Delivery Date', 'Delivered to Client Date', 'Delivery Recorded Date']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. Helper function to parse non-numeric strings (e.g., 'Freight Included', 'See ASN-...')
def coerce_to_numeric(val):
    if pd.isna(val):
        return np.nan
    val_clean = str(val).strip().replace('$', '').replace(',', '')
    try:
        return float(val_clean)
    except ValueError:
        return np.nan  # Turns text descriptors into Nulls for safe aggregations

# Apply coercion across required structural columns
df['Weight_Clean'] = df['Weight (Kilograms)'].apply(coerce_to_numeric)
df['Freight_Cost_USD_Clean'] = df['Freight Cost (USD)'].apply(coerce_to_numeric)
df['Unit_Price_Clean'] = df['Unit Price'].apply(coerce_to_numeric)
df['Line_Item_Value_Clean'] = df['Line Item Value'].apply(coerce_to_numeric)
df['Insurance_Clean'] = df['Line Item Insurance (USD)'].apply(coerce_to_numeric)

# 3. Handle Missing Values using business logic
df['Freight_Cost_USD_Clean'] = df['Freight_Cost_USD_Clean'].fillna(0)
df['Weight_Clean'] = df['Weight_Clean'].fillna(df['Weight_Clean'].median())
df['Insurance_Clean'] = df['Insurance_Clean'].fillna(0)

# 4. Calculate exact delivery performance delay
df['Delivery_Delay'] = (df['Delivered to Client Date'] - df['Scheduled Delivery Date']).dt.days

print("Data Cleaning Complete. All anomalies handled successfully.")

# ==========================================
# STEP 3: GENERATING VISUALIZATIONS
# ==========================================
print("\n--- STEP 3: GENERATING PYTHON VISUALIZATIONS ---")

# Chart 1: Country vs Shipments (Top 10)
plt.figure(figsize=(10, 5))
df['Country'].value_counts().head(10).plot(kind='bar', color='royalblue')
plt.title('Top 10 Countries by Supply Chain Shipment Volume')
plt.xlabel('Country')
plt.ylabel('Total Shipments')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('python_chart_country_shipments.png')
plt.close()

# Chart 2: Shipment Mode Distribution
plt.figure(figsize=(7, 7))
df['Shipment Mode'].value_counts(dropna=False).plot(kind='pie', autopct='%1.1f%%', colors=['#4F81BD', '#C0504D', '#9BBB59', '#20B2AA', '#778899'])
plt.title('Logistics Distribution by Shipment Mode')
plt.ylabel('')
plt.savefig('python_chart_shipment_mode.png')
plt.close()

# Chart 3: Delivery Trend Over Time (Using 'ME' for Pandas 3.0+)
plt.figure(figsize=(11, 5))
df.set_index('Delivered to Client Date').resample('ME')['Line_Item_Value_Clean'].sum().plot(linewidth=2.5, color='darkorange')
plt.title('Temporal Cargo Movement: Total Line Item Value Shipped Over Time')
plt.xlabel('Delivery Timeline')
plt.ylabel('Total Value ($)')
plt.tight_layout()
plt.savefig('python_chart_delivery_trend.png')
plt.close()

print("All 3 charts saved to your root project folder.")

# ==========================================
# STEP 4: EXPORT COMPLETELY CLEANED DATA
# ==========================================
print("\n--- STEP 4: EXPORTING CLEANED DATASET ---")
df.to_csv("SCMS_Delivery_History_Cleaned.csv", index=False)
print("--- SUCCESS: 'SCMS_Delivery_History_Cleaned.csv' is ready for Power BI! ---")

--- STEP 1: LOADING DATASET ---
Dataset Shape: (10325, 33)

--- STEP 2: EXECUTING DATA CLEANING ---
Data Cleaning Complete. All anomalies handled successfully.

--- STEP 3: GENERATING PYTHON VISUALIZATIONS ---
All 3 charts saved to your root project folder.

--- STEP 4: EXPORTING CLEANED DATASET ---
--- SUCCESS: 'SCMS_Delivery_History_Cleaned.csv' is ready for Power BI! ---
